In [419]:
import rebound
import reboundx
import numpy as np
import matplotlib.pyplot as plt
import astropy.constants as constants
import astropy.units as units

In [420]:
rebound.__version__

'4.4.6'

In [421]:
date='2000-01-01 00:00'
sim = rebound.Simulation()
sim.add('Sun', date=date, hash='sun')
sim.add('Mercury', date=date)
sim.add('Venus', date=date)
sim.add('Earth', date=date, hash='earth')
sim.add('Mars', date=date)
sim.add('Jupiter', date=date)
sim.add('Saturn', date=date)
sim.add('Uranus', date=date)
sim.add('Neptune', date=date)
sim.add('Pluto', date=date)
sim.move_to_com()
sim.convert_particle_units('AU', 'year', 'Msun')
# sim.save_to_file('r01.bin')
ps = sim.particles

Searching NASA Horizons for 'Sun'... 
Found: Sun (10) 
Searching NASA Horizons for 'Mercury'... 
Found: Mercury Barycenter (199) (chosen from query 'Mercury')
Searching NASA Horizons for 'Venus'... 
Found: Venus Barycenter (299) (chosen from query 'Venus')
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')
Searching NASA Horizons for 'Mars'... 
Found: Mars Barycenter (4) (chosen from query 'Mars')
Searching NASA Horizons for 'Jupiter'... 
Found: Jupiter Barycenter (5) (chosen from query 'Jupiter')
Searching NASA Horizons for 'Saturn'... 
Found: Saturn Barycenter (6) (chosen from query 'Saturn')
Searching NASA Horizons for 'Uranus'... 
Found: Uranus Barycenter (7) (chosen from query 'Uranus')
Searching NASA Horizons for 'Neptune'... 
Found: Neptune Barycenter (8) (chosen from query 'Neptune')


In [422]:
tmax = -3.5e9
Nsnaps = 1e3
interval = int(abs(tmax/Nsnaps))

# sim= rebound.Simulation('r01.bin')
sim.integrator = "WHCKL" 
sim.ri_whfast.safe_mode = False
sim.ri_whfast.corrector = 17
sim.ri_whfast.keep_unsynchronized=True
sim.dt = 4.062/365.25
sim.save_to_file('r01.bin', interval=interval, delete_file=True)

In [ ]:
# times = [n*interval for n in Nsnaps]
# Mstars = [M(t) for t in times]
# rs = [r(t) for t in times]
# for i, (time, M, r) in enumerate(zip(times, Mstars, rs)):
    

In [423]:
print("Snapshot length is:",interval/1e3, "ky")

Snapshot length is: 35.0 ky


In [424]:
Lx, Ly, Lz = sim.angular_momentum()
Lvec = [Lx,Ly,Lz]

In [425]:
rebx = reboundx.Extras(sim)
gr = rebx.load_force('gr_potential')
rebx.add_force(gr)
gr.params['c'] = 63240 # speed of light in AU/yr

cf = rebx.load_force("quadrupole")
rebx.add_force(cf)

earth_m = ps['earth'].m/1.0123000370338813
f = 0.8525
mu_eff = f*(1*0.0123000370338813*(earth_m)**2)/(1.0123000370338813*earth_m)
R = 0.0025696
ps['earth'].params["Rcentral"] = R
ps['earth'].params["mu_effcentral"] = mu_eff

gh = rebx.load_force("gravitational_harmonics")
rebx.add_force(gh)
J2 = 2.25*1e-7
sim.particles['sun'].params["J2"] = J2

inc_sun = np.radians(7.155) # rad
Omega_sun = np.radians(75.594) # rad
R_eq_sun = (constants.R_sun.to(au)).value

spin_axis_vector = [np.sin(inc_sun) * np.sin(Omega_sun), -np.sin(inc_sun) * np.cos(Omega_sun), np.cos(inc_sun)]
sim.particles['sun'].params["Omega"] = spin_axis_vector
sim.particles['sun'].params["R_eq"] = R_eq_sun

R_e = (constants.R_earth.to(au)).value
ratio = R / R_e
M0 = ps['sun'].m

In [426]:
print("Radius of the Sun is:", R_eq_sun, "AU")
print("Radius of the Earth is: ", R_e, "AU")
print("Speed of light in AU/yr is: ", gr.params['c'])

#offset from ecliptic plane ECLIPJ2000
zaxis = [0.,0.,1.0]
offset = np.degrees(np.arccos(np.dot(zaxis/np.linalg.norm(zaxis), spin_axis_vector/np.linalg.norm(spin_axis_vector))))
print("offset from ecliptic plane ECLIPJ2000 is: ", offset, "deg")

#offset from invariable plane Lvec
offset = np.degrees(np.arccos(np.dot(Lvec/np.linalg.norm(Lvec), spin_axis_vector/np.linalg.norm(spin_axis_vector))))
print("offset from invariable plane is: ", offset, "deg")

Radius of the Sun is: 0.004650467260962157 AU
Radius of the Earth is:  4.263496512454037e-05 AU
Speed of light in AU/yr is:  63240.0
offset from ecliptic plane ECLIPJ2000 is:  7.154999999999989 deg
offset from invariable plane is:  5.875673975329417 deg


In [427]:
%%time

times = np.linspace(0., tmax, Nsnaps)
rate = (-7.e14)
Rchive = np.zeros(Nsnaps)
mass = np.zeros(Nsnaps)

for i, time in enumerate(times):
    sim.integrate(time)
    sim.particles[0].m = M0*np.exp(time / rate)
    mass[i] = sim.particles[0].m
    r = (ratio-((5.14/ 1.e9)*(abs(time))))*R_e
    ps['earth'].params["Rcentral"] = r
    Rchive[i] = ps['earth'].params["Rcentral"]

KeyboardInterrupt: 